<a href="https://colab.research.google.com/github/salalisiasa-debug/-2026/blob/main/lab_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 2
## Колоночное хранение, партиционирование и бенчмаркинг Parquet

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование
**Профиль:** «Информатика и дополнительное образование (робототехника)»
**Этап конвейера:** `Collect → Store → Process`
**Среда:** Google Colab · PySpark 4.2.0 · `local[*]` · Parquet · Snappy / ZSTD / GZIP

---

### Цель работы

Освоить колоночное хранение телеметрии в формате Parquet, физическое партиционирование набора
и экспериментальное сравнение объёма и времени чтения CSV и Parquet. Понять, за счёт чего
возникает выигрыш и когда он исчезает.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в GitHub или Colab.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Три физические версии набора: CSV, Parquet, partitioned Parquet | 2 |
| 2 | Размеры, коэффициент сжатия, влияние энтропии и выбора кодека | 2 |
| 3 | Воспроизводимый бенчмарк чтения: ≥ 3 повтора, медиана, честное сравнение | 2 |
| 4 | Column pruning, partition pruning, проблема мелких файлов | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения — пользуйтесь `help()` и документацией Spark.
> Ячейки с `assert` — самопроверка: если проверка прошла, задание засчитано. Числовые значения
> времени зависят от runtime Colab, поэтому оценивается методика измерения, а не абсолютные цифры.

### Источники

* Balapriya C. `data-science-tutorials` — <https://github.com/balapriyac/data-science-tutorials>
  (`pyspark/pyspark_write_parquet.ipynb`)
* Piotr Szul. `spark-tutorial` — <https://github.com/piotrszul/spark-tutorial>
* Apache Parquet — <https://parquet.apache.org/docs/>
* Spark. Parquet Files — <https://spark.apache.org/docs/latest/sql-data-sources-parquet.html>

---
## Задание 0. Подготовка среды и данных

### Минимальные теоретические сведения

**Колоночное хранение.** Parquet размещает рядом значения одной колонки, а не одной записи.
Отсюда два следствия: однородные данные сжимаются в разы лучше, а запрос, которому нужны две
колонки из десяти, читает только эти две — это **column pruning**.

**Партиционирование.** `partitionBy("robot_id")` раскладывает данные по каталогам вида
`robot_id=robot-7/`. Запрос с фильтром по этой колонке пропускает лишние каталоги целиком —
это **partition pruning**. Значение ключа при этом физически не хранится в файлах: оно
закодировано в имени каталога.

**Метрики работы:**

коэффициент уменьшения объёма  $C = S_{CSV} / S_{Parquet}$

ускорение чтения  $Speedup = T_{CSV} / T_{Parquet}$

Результаты зависят от runtime Colab и файлового кэша, поэтому каждое измерение повторяется
не менее трёх раз, а итогом берётся **медиана** (она устойчивее среднего к единичному выбросу
при первом «холодном» чтении).

In [ ]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

In [ ]:
import shutil
import time
import statistics
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)

ROOT = Path("/content/lab02_storage")
if not Path("/content").exists():          # запуск вне Colab
    ROOT = Path("./lab02_storage")

CSV_DIR         = ROOT / "csv"
PARQUET_DIR     = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог хранения:", ROOT)

Spark version: 4.2.0
Каталог хранения: /content/lab02_storage


### Генерация телеметрии

Набор порождается детерминированно, внешний датасет не нужен. Обратите внимание на колонку
`payload`: это SHA-256 от идентификатора, то есть **высокоэнтропийная** строка без повторов.
Она добавлена намеренно — в части 2 мы посмотрим, что она делает со сжатием.

DataFrame кэшируется: иначе Spark пересчитает его заново при каждой из трёх записей.

In [ ]:
N = 300_000

df = (
    spark.range(N)
    .withColumn("robot_id",    F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor",      F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms",       (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration",   F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload",     F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
    .cache()
)

print("Строк:", df.count())
df.printSchema()
df.show(3, truncate=False)

Строк: 300000
root
 |-- robot_id: string (nullable = true)
 |-- sensor: string (nullable = false)
 |-- ts_ms: long (nullable = false)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)

+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot_id|sensor|ts_ms        |temperature|vibration          |payload                                                         |
+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot-0 |imu   |1720000000000|40.0       |0.0                |5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|
|robot-1 |motor |1720000000100|40.08      |0.068617294923393  |6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|
|robot-2 |imu   |1720000000200|40.16      |0.13693862262231407|d4735e3a265e16eee03f59718b9b5d0301

---
# Часть 1. Три физические версии набора (2 балла)

Один и тот же DataFrame нужно записать тремя способами и дальше сравнивать их между собой.

## Упражнение 1.1. CSV

Запишите `df` в CSV в каталог `CSV_DIR`. Режим записи — `overwrite`, заголовок включён
(`option("header", True)`).

In [ ]:
# TODO: df.write ... .csv(str(CSV_DIR))
df.write.mode("overwrite").option("header",True).csv(str(CSV_DIR))
files = sorted(p.name for p in CSV_DIR.iterdir())
print("Файлов в каталоге CSV:", len(files))
print(files[:5])
assert CSV_DIR.exists() and any(p.suffix == ".csv" for p in CSV_DIR.iterdir())

Файлов в каталоге CSV: 6
['._SUCCESS.crc', '.part-00000-e970d83a-bd75-420c-bff6-20a2bbf8ab2c-c000.csv.crc', '.part-00001-e970d83a-bd75-420c-bff6-20a2bbf8ab2c-c000.csv.crc', '_SUCCESS', 'part-00000-e970d83a-bd75-420c-bff6-20a2bbf8ab2c-c000.csv']


> **Обратите внимание.** Spark пишет не один файл, а каталог: по одному файлу `part-*` на каждую
> partition плюс маркер `_SUCCESS`. Так устроены все распределённые форматы — единый файл нельзя
> писать параллельно с нескольких узлов.

## Упражнение 1.2. Parquet со сжатием Snappy

Запишите тот же `df` в Parquet в каталог `PARQUET_DIR`, явно указав кодек
`option("compression", "snappy")`.

In [ ]:
# TODO: df.write ... .parquet(str(PARQUET_DIR))
df.write.mode("overwrite").option("compression","snappy").parquet(str(PARQUET_DIR))
parts = [p.name for p in PARQUET_DIR.iterdir() if p.suffix == ".parquet"]
print("Файлов parquet:", len(parts))
print(parts[:3])
assert len(parts) > 0
assert all("snappy" in p for p in parts), "В имени файла должен быть виден кодек"

Файлов parquet: 2
['part-00000-4b111f53-02a3-4157-a76c-4c709731beaf-c000.snappy.parquet', 'part-00001-4b111f53-02a3-4157-a76c-4c709731beaf-c000.snappy.parquet']


## Упражнение 1.3. Partitioned Parquet

Запишите набор в `PARTITIONED_DIR` с физическим партиционированием по `robot_id`
(`partitionBy`). Затем выведите список созданных каталогов и проверьте, что их ровно 12 —
по числу роботов.

In [ ]:
# TODO: df.write ... .partitionBy(...).parquet(str(PARTITIONED_DIR))
df.write.mode("overwrite").partitionBy("robot_id").parquet(str(PARTITIONED_DIR))
part_dirs = sorted(p.name for p in PARTITIONED_DIR.iterdir() if p.is_dir())
print("Каталогов:", len(part_dirs))
print(part_dirs[:4])

assert len(part_dirs) == 12
assert all(d.startswith("robot_id=") for d in part_dirs)

Каталогов: 12
['robot_id=robot-0', 'robot_id=robot-1', 'robot_id=robot-10', 'robot_id=robot-11']


## Упражнение 1.4

Прочитайте один файл из каталога `robot_id=robot-7` напрямую (`spark.read.parquet` по полному
пути к каталогу партиции) и посмотрите его схему. Есть ли в ней колонка `robot_id`?
Ответьте письменно, почему.

In [ ]:
one_partition = spark.read.parquet(str(PARTITIONED_DIR / "robot_id=robot-7"))
one_partition.printSchema()

ANSWER_1_4 = """
Колонки robot_id внутри файла нет, потому что Spark хранит её значение
в названии папки, например robot_id=robot-7. Поэтому robot-7 не записывается
много раз в каждой строке файла. Когда Spark читает весь каталог, он видит
названия папок и сам восстанавливает колонку robot_id
"""
print(ANSWER_1_4)

assert "robot_id" not in one_partition.columns

root
 |-- sensor: string (nullable = true)
 |-- ts_ms: long (nullable = true)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)


Колонки robot_id внутри файла нет, потому что Spark хранит её значение
в названии папки, например robot_id=robot-7. Поэтому robot-7 не записывается
много раз в каждой строке файла. Когда Spark читает весь каталог, он видит
названия папок и сам восстанавливает колонку robot_id



---
# Часть 2. Объём хранения (2 балла)

## Упражнение 2.1. Размеры и коэффициент сжатия

Напишите функцию `dir_size(path)`, возвращающую суммарный размер всех файлов каталога в байтах
(каталог вложенный, поэтому нужен рекурсивный обход — подсказка: `Path.rglob`).

Выведите таблицу размеров трёх версий и рассчитайте коэффициент
$C = S_{CSV} / S_{Parquet}$.

In [ ]:
def dir_size(path: Path) -> int:
    """Суммарный размер всех файлов внутри каталога, включая вложенные."""
    # TODO
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


sizes = {
    "CSV":                 dir_size(CSV_DIR),
    "Parquet":             dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}
for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:7.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"\nКоэффициент C = S_CSV / S_Parquet = {compression_ratio:.2f}x")

assert sizes["Parquet"] < sizes["CSV"], "Parquet должен быть компактнее CSV"
assert compression_ratio > 1.0

CSV                   :   33.96 MiB
Parquet               :   22.67 MiB
Partitioned Parquet   :   22.85 MiB

Коэффициент C = S_CSV / S_Parquet = 1.50x


## Упражнение 2.2. Энтропия и сжимаемость

Коэффициент из предыдущего упражнения получился скромным. Причина — колонка `payload`:
это SHA-256, то есть практически случайная строка. Данные с высокой энтропией не сжимаются
принципиально: в них нет повторов, которые можно было бы закодировать короче.

Запишите **тот же набор без колонки `payload`** в CSV и в Parquet, посчитайте коэффициент
заново и сравните с предыдущим.

In [ ]:
NOPAY_CSV = ROOT / "csv_nopayload"
NOPAY_PQ  = ROOT / "parquet_nopayload"

df_nopay = df.drop("payload")

# TODO: запишите df_nopay в CSV (с заголовком) и в Parquet (snappy)
df_nopay.write.mode("overwrite").option("header", True).csv(str(NOPAY_CSV))
df_nopay.write.mode("overwrite").option("compression","snappy").parquet(str(NOPAY_PQ))

sizes_nopay = {"CSV": dir_size(NOPAY_CSV), "Parquet": dir_size(NOPAY_PQ)}
ratio_nopay = sizes_nopay["CSV"] / sizes_nopay["Parquet"]

print(f"Без payload: CSV {sizes_nopay['CSV']/1024/1024:.2f} MiB, "
      f"Parquet {sizes_nopay['Parquet']/1024/1024:.2f} MiB")
print(f"Коэффициент без payload: {ratio_nopay:.2f}x  (с payload было {compression_ratio:.2f}x)")

ANSWER_2_2 = """
Колонка payload содержит значения SHA-256, которые практически не повторяются,
поэтому алгоритмам сжатия сложно найти в них повторяющиеся последовательности.
В остальных колонках, например robot_id и sensor, одинаковые значения встречаются
много раз, поэтому Parquet может эффективно использовать словарное кодирование и другие
способы сжатия. После удаления payload доля хорошо сжимаемых данных увеличивается,
поэтому общий коэффициент сжатия становится выше. Следовательно, эффективность Parquet
зависит не только от формата файла, но и от структуры и повторяемости самих данных.
"""
print(ANSWER_2_2)

assert ratio_nopay > compression_ratio, "Без высокоэнтропийной колонки выигрыш должен вырасти"

Без payload: CSV 15.22 MiB, Parquet 3.61 MiB
Коэффициент без payload: 4.21x  (с payload было 1.50x)

Колонка payload содержит значения SHA-256, которые практически не повторяются,
поэтому алгоритмам сжатия сложно найти в них повторяющиеся последовательности. 
В остальных колонках, например robot_id и sensor, одинаковые значения встречаются
много раз, поэтому Parquet может эффективно использовать словарное кодирование и другие 
способы сжатия. После удаления payload доля хорошо сжимаемых данных увеличивается, 
поэтому общий коэффициент сжатия становится выше. Следовательно, эффективность Parquet 
зависит не только от формата файла, но и от структуры и повторяемости самих данных.



## Упражнение 2.3. Сравнение кодеков

Запишите `df_nopay` в Parquet четырьмя кодеками: `uncompressed`, `snappy`, `gzip`, `zstd`.
Для каждого измерьте время записи и итоговый размер. Заполните словарь `codec_stats`
в формате `{кодек: (секунды, байты)}`.

In [ ]:
codec_stats = {}

for codec in ["uncompressed", "snappy", "gzip", "zstd"]:
    out = ROOT / f"pq_{codec}"
    # TODO: замерьте время записи df_nopay с этим кодеком и размер каталога
    t0=time.perf_counter()
    df_nopay.write.mode("overwrite").option("compression",codec).parquet(str(out))
    elapsed=time.perf_counter()-t0
    codec_stats[codec]=(elapsed,dir_size(out))
    pass

print(f"{'кодек':<14}{'время, с':>10}{'размер, MiB':>14}{'C к uncompressed':>20}")
base = codec_stats["uncompressed"][1]
for codec, (t, s) in codec_stats.items():
    print(f"{codec:<14}{t:>10.2f}{s/1024/1024:>14.2f}{base/s:>20.2f}")

assert set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"}
assert codec_stats["gzip"][1] < codec_stats["snappy"][1], "gzip должен сжимать сильнее snappy"

кодек           время, с   размер, MiB    C к uncompressed
uncompressed        0.44          5.09                1.00
snappy              0.36          3.61                1.41
gzip                0.44          2.70                1.89
zstd                0.31          2.61                1.95


**Вопрос 2.4.** Почему Snappy используется по умолчанию, хотя gzip сжимает заметно сильнее?
Ответьте письменно.

In [ ]:
ANSWER_2_4 = """
Snappy используется по умолчанию, потому что он очень быстро сжимает и распаковывает данные.
В Parquet файлы обычно читаются много раз, поэтому скорость чтения важнее максимального сжатия.
Gzip сжимает сильнее, но требует больше времени и нагрузки на процессор. Поэтому Snappy считается
удобным компромиссом между размером файла и скоростью работы.
"""
print(ANSWER_2_4)


Snappy используется по умолчанию, потому что он очень быстро сжимает и распаковывает данные. 
В Parquet файлы обычно читаются много раз, поэтому скорость чтения важнее максимального сжатия. 
Gzip сжимает сильнее, но требует больше времени и нагрузки на процессор. Поэтому Snappy считается 
удобным компромиссом между размером файла и скоростью работы.



---
# Часть 3. Бенчмарк чтения (2 балла)

## Упражнение 3.1. Функция измерения

Напишите функцию `benchmark_read(fmt, path, predicate=None, repeats=3, schema=None)`, которая:

1. `repeats` раз читает набор из `path` в формате `fmt` (`"csv"` или `"parquet"`);
2. при заданном `predicate` применяет фильтр;
3. выполняет действие `count()` — **без него из-за ленивости ничего не прочитается**;
4. возвращает кортеж `(медиана времени, число строк)`.

Для CSV используйте `option("header", True)`; параметр `schema` пока оставьте
необязательным — он понадобится в упражнении 3.3.

In [ ]:
def benchmark_read(fmt, path, predicate=None, repeats=3, schema=None):
    """Возвращает (медианное время чтения, число строк)."""
    times, counts = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        # TODO: прочитайте данные нужного формата, примените predicate, вызовите count()
        if fmt == "csv":
          reader=spark.read.option("header", True)
          if schema is not None:
            reader = reader.schema(schema)
          else:
            reader=reader.option("inferSchema", True)
          x=reader.csv(str(path))
        else:
            x=spark.read.parquet(str(path))
        if predicate is not None:
         x=x.filter(predicate)
        counts.append(x.count())
        times.append(time.perf_counter()-t0)
    return statistics.median(times), counts[-1]
    return statistics.median(times), counts[-1]


t, n = benchmark_read("parquet", PARQUET_DIR, repeats=3)
print(f"Чтение всего Parquet: {t:.3f} с, строк {n}")
assert n == N

Чтение всего Parquet: 0.353 с, строк 300000


## Упражнение 3.2. Сравнение трёх версий

Выполните одинаковый запрос `robot_id == "robot-7"` к каждой из трёх версий набора, минимум
по три повтора. Рассчитайте ускорение относительно CSV.

In [ ]:
predicate = F.col("robot_id") == "robot-7"

csv_t,  csv_n  = benchmark_read("csv",     CSV_DIR,         predicate)
pq_t,   pq_n   = benchmark_read("parquet", PARQUET_DIR,     predicate)
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, predicate)

print(f"CSV                 : {csv_t:6.3f} с, строк {csv_n}")
print(f"Parquet             : {pq_t:6.3f} с, строк {pq_n}")
print(f"Partitioned Parquet : {part_t:6.3f} с, строк {part_n}")
print()
print(f"Speedup CSV -> Parquet             : {csv_t / pq_t:.2f}x")
print(f"Speedup CSV -> Partitioned Parquet : {csv_t / part_t:.2f}x")

assert csv_n == pq_n == part_n == N // 12, "Все три версии должны вернуть одинаковое число строк"

CSV                 :  3.013 с, строк 25000
Parquet             :  0.289 с, строк 25000
Partitioned Parquet :  0.309 с, строк 25000

Speedup CSV -> Parquet             : 10.42x
Speedup CSV -> Partitioned Parquet : 9.74x


## Упражнение 3.3. Честное сравнение

В упражнении 3.1 CSV читался с `inferSchema=True`. Это удобно, но нечестно по отношению к CSV:
чтобы вывести типы, Spark делает **дополнительный полный проход** по файлу. Получается, что
мы сравниваем Parquet с CSV, прочитанным дважды.

Повторите замер CSV с **явно заданной схемой** — возьмите её у уже прочитанного Parquet
(`spark.read.parquet(...).schema`) — и сравните три числа: CSV с выводом типов, CSV со схемой
и Parquet.

In [ ]:
explicit_schema = spark.read.parquet(str(PARQUET_DIR)).schema

# TODO: замерьте чтение CSV с явной схемой (schema=explicit_schema)
csv_schema_t, csv_schema_n = benchmark_read("csv",CSV_DIR,predicate,schema=explicit_schema)


print(f"CSV + inferSchema : {csv_t:6.3f} с")
print(f"CSV + явная схема : {csv_schema_t:6.3f} с")
print(f"Parquet           : {pq_t:6.3f} с")
print(f"\nЧестный speedup CSV -> Parquet: {csv_schema_t / pq_t:.2f}x")

ANSWER_3_3 = """
Выигрыш Parquet уменьшилось, потому что раньше CSV дополнительно проходил
по файлу для определения типов данных. Когда для CSV явно задали схему,
сравнение стало честнее, так как оба формата читаются в более равных условиях.
Parquet всё равно работает быстрее, потому что занимает меньше места и хранит
данные в бинарном виде, без разбора текста. Если нужны только отдельные столбцы,
Parquet также может читать только их, а не весь файл.
"""
print(ANSWER_3_3)

assert csv_schema_n == csv_n
assert csv_schema_t < csv_t, "Без inferSchema чтение CSV должно ускориться"

CSV + inferSchema :  3.013 с
CSV + явная схема :  0.689 с
Parquet           :  0.289 с

Честный speedup CSV -> Parquet: 2.38x

Выигрыш Parquet уменьшилось, потому что раньше CSV дополнительно проходил 
по файлу для определения типов данных. Когда для CSV явно задали схему, 
сравнение стало честнее, так как оба формата читаются в более равных условиях. 
Parquet всё равно работает быстрее, потому что занимает меньше места и хранит 
данные в бинарном виде, без разбора текста. Если нужны только отдельные столбцы, 
Parquet также может читать только их, а не весь файл.



---
# Часть 4. Pruning и мелкие файлы (2 балла)

## Упражнение 4.1. Partition pruning

Выполните `explain(mode="formatted")` для запроса к partitioned Parquet с фильтром по
`robot_id` и найдите в плане строку `PartitionFilters`.

In [ ]:
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [temperature#5843, robot_id#5846]
Batched: true
Location: InMemoryFileIndex [file:/content/lab02_storage/parquet_partitioned]
PartitionFilters: [isnotnull(robot_id#5846), (robot_id#5846 = robot-7)]
ReadSchema: struct<temperature:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [temperature#5843, robot_id#5846]

(3) Project [codegen id : 1]
Output [2]: [robot_id#5846, temperature#5843]
Input [2]: [temperature#5843, robot_id#5846]




In [ ]:
ANSWER_4_1 = """
 Что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
    PartitionFilters показывает, что Spark читает только нужную партицию, например
  robot_id = robot-7, а остальные папки пропускает. PushedFilters — это
  обычные фильтры по данным внутри Parquet, которые помогают не читать лишние блоки.
  ReadSchema показывает, какие столбцы реально считываются с диска.
  В итоге Spark экономит время сразу двумя способами: не читает лишние
  партиции и не загружает ненужные столбцы.
"""
print(ANSWER_4_1)


 Что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
    PartitionFilters показывает, что Spark читает только нужную партицию, например 
  robot_id = robot-7, а остальные папки пропускает. PushedFilters — это 
  обычные фильтры по данным внутри Parquet, которые помогают не читать лишние блоки. 
  ReadSchema показывает, какие столбцы реально считываются с диска. 
  В итоге Spark экономит время сразу двумя способами: не читает лишние 
  партиции и не загружает ненужные столбцы.   



## Упражнение 4.2. Column pruning в цифрах

Сравните время двух запросов к обычному (непартиционированному) Parquet:

* агрегация **только** по `temperature`;
* агрегация по `temperature` **и** по тяжёлой высокоэнтропийной колонке `payload`.

Второй запрос вынуждает Spark поднять с диска колонку, которая занимает большую часть объёма
файла. Каждый запрос — минимум три повтора, берите медиану.

Дополнительно выведите `explain` обоих вариантов и сравните строки `ReadSchema`.

In [ ]:
def timed(fn, repeats=3):
    """Медианное время выполнения fn()."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


def read_with_payload():
    # TODO: agg по temperature И по payload (например, F.max("payload"))
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature"),F.max("payload")).collect())


def read_one_column():
    # TODO: agg только по temperature
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature")).collect())


t_all = timed(read_with_payload)
t_one = timed(read_one_column)
print(f"temperature + payload: {t_all:.3f} с")
print(f"только temperature   : {t_one:.3f} с")
print(f"Column pruning дал ускорение в {t_all / t_one:.2f}x")

spark.read.parquet(str(PARQUET_DIR)).select("temperature").agg(F.avg("temperature")).explain()

ANSWER_4_2 = """
Разница во времени возникает из-за того, сколько данных Spark реально
считывает с диска. Если в запросе используется колонка payload, Parquet
должен прочитать её целиком, а она занимает значительную часть файла,
поэтому запрос выполняется дольше. Если payload в запросе не нужен, Spark
благодаря column pruning читает только необходимые столбцы и пропускает
остальные. Это особенно важно для Parquet, потому что он хранит данные по
колонкам. Поэтому на широких таблицах выбор только нужных полей может
заметно уменьшить объём чтения и ускорить выполнение запроса.
"""
print(ANSWER_4_2)

temperature + payload: 0.451 с
только temperature   : 0.308 с
Column pruning дал ускорение в 1.46x
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(temperature#5172)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=2294]
      +- HashAggregate(keys=[], functions=[partial_avg(temperature#5172)])
         +- FileScan parquet [temperature#5172] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab02_storage/parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<temperature:double>



Разница во времени возникает из-за того, сколько данных Spark реально 
считывает с диска. Если в запросе используется колонка payload, Parquet 
должен прочитать её целиком, а она занимает значительную часть файла, 
поэтому запрос выполняется дольше. Если payload в запросе не нужен, Spark 
благодаря column pruning читает только необходимые столбцы и пропускает 
остальные. Это особенно

## Упражнение 4.3. Проблема мелких файлов

Партиционирование выглядит бесплатным улучшением, но это не так. Запишите набор,
партиционированный по искусственному ключу высокой кардинальности `ts_bucket` (200 значений),
и сравните с партиционированием по `robot_id` (12 значений):

* число созданных файлов;
* суммарный объём;
* средний размер одного файла;
* время чтения всего набора.

In [ ]:
MANY_DIR = ROOT / "parquet_many_partitions"

df_buckets = df_nopay.withColumn(
    "ts_bucket", ((F.col("ts_ms") / 100) % 200).cast("int")
)
# TODO: запишите df_buckets в MANY_DIR с partitionBy("ts_bucket")
df_buckets.write.mode("overwrite").partitionBy("ts_bucket").parquet(str(MANY_DIR))


def file_stats(path: Path):
    """Возвращает (число файлов .parquet, суммарный размер, средний размер)."""
    # TODO
    files=[p for p in path.rglob("*.parquet") if p.is_file()]
    total=sum(p.stat().st_size for p in files)
    return len(files), total, total / max(len(files),1)



n_few,  s_few,  avg_few  = file_stats(PARTITIONED_DIR)
n_many, s_many, avg_many = file_stats(MANY_DIR)

print(f"{'вариант':<28}{'файлов':>9}{'объём, MiB':>13}{'средний файл, KiB':>20}")
print(f"{'partitionBy(robot_id) [12]':<28}{n_few:>9}{s_few/1024/1024:>13.2f}{avg_few/1024:>20.1f}")
print(f"{'partitionBy(ts_bucket) [200]':<28}{n_many:>9}{s_many/1024/1024:>13.2f}{avg_many/1024:>20.1f}")

t_few  = timed(lambda: spark.read.parquet(str(PARTITIONED_DIR)).count())
t_many = timed(lambda: spark.read.parquet(str(MANY_DIR)).count())
print(f"\nЧтение целиком: 12 партиций {t_few:.3f} с, 200 партиций {t_many:.3f} с")

assert n_many > n_few
assert avg_many < avg_few

вариант                        файлов   объём, MiB   средний файл, KiB
partitionBy(robot_id) [12]         24        22.67               967.5
partitionBy(ts_bucket) [200]      400         4.47                11.4

Чтение целиком: 12 партиций 0.241 с, 200 партиций 3.068 с


## Упражнение 4.4. Технический вывод

Ответьте письменно:

1. Почему слишком большое число мелких partition ухудшает эффективность хранения и чтения?
2. По какому признаку выбирать колонку для `partitionBy`?
3. Когда партиционирование по `robot_id` в нашей задаче окажется бесполезным?

In [ ]:
ANSWER_4_4 = """
1. Мелкие файлы в Parquet неудобны, потому что каждый файл имеет свои служебные
данные и требует отдельного открытия и обработки. Если файлов становится очень
много, эти дополнительные расходы начинают занимать заметное время и место.
Поэтому лучше хранить данные в более крупных файлах, а не разбивать их на тысячи маленьких.
2. Колонку для partitionBy стоит выбирать так, чтобы по ней часто выполнялась
фильтрация и чтобы значений было не слишком много. Если уникальных значений
слишком много, получится большое количество мелких папок и файлов. Хороший
вариант — поле, по которому данные часто ищут и которое делит набор на достаточно крупные части.
3. Партиционирование по robot_id бесполезно, если запросы обычно читают данные
сразу по всем роботам. В таком случае Spark всё равно будет просматривать все
партиции, и ускорения не будет. Также плохо, если данные распределены неравномерно
или роботов становится слишком много, потому что тогда появляются слишком маленькие
или слишком многочисленные партиции.
"""
print(ANSWER_4_4)


1. Мелкие файлы в Parquet неудобны, потому что каждый файл имеет свои служебные 
данные и требует отдельного открытия и обработки. Если файлов становится очень 
много, эти дополнительные расходы начинают занимать заметное время и место. 
Поэтому лучше хранить данные в более крупных файлах, а не разбивать их на тысячи маленьких.
2. Колонку для partitionBy стоит выбирать так, чтобы по ней часто выполнялась
фильтрация и чтобы значений было не слишком много. Если уникальных значений 
слишком много, получится большое количество мелких папок и файлов. Хороший 
вариант — поле, по которому данные часто ищут и которое делит набор на достаточно крупные части.
3. Партиционирование по robot_id бесполезно, если запросы обычно читают данные 
сразу по всем роботам. В таком случае Spark всё равно будет просматривать все 
партиции, и ускорения не будет. Также плохо, если данные распределены неравномерно 
или роботов становится слишком много, потому что тогда появляются слишком маленькие 
или слишком м

---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы.

**Требования к заполнению.** Формулировки конкретные. «Расскажу про Parquet» не засчитывается.
Нужны измеримый результат, критерий успеха и честно названные ограничения.

**Подсказка по содержанию.** Эксперимент «CSV против Parquet» удобен для школы тем, что результат
виден глазами: два каталога с одними и теми же данными занимают разный объём, и это можно
измерить без единой строки распределённого кода — хватит `du -h`. Аналогия для урока: если
нужна только температура, читается только «стопка температур», а не все листы журнала целиком.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | Сравнение эффективности форматов CSV и Parquet |
| **Целевая аудитория** | 10–11 класс / кружок робототехники  |
| **Исследовательский вопрос** | Как изменяются размер данных и время их чтения при хранении одного и того же набора телеметрии в форматах CSV и Parquet, и как выбор только нужных столбцов и партиций влияет на время чтения Parquet? |
| **Источник данных** | Синтетически сгенерированный в PySpark набор телеметрии роботов: 300 000 строк и 6 колонок (robot_id, sensor, ts_ms, temperature, vibration, payload с SHA-256) |
| **Инженерная концепция, сохраняемая без упрощения** | Колоночное хранение данных: Parquet хранит значения каждого столбца вместе, поэтому при чтении, например, двух столбцов из шести остальные можно не загружать. Также сохраняется принцип партиционирования: данные разделяются по каталогам вида robot_id=robot-7, поэтому при фильтрации по robot_id Spark может сразу пропускать ненужные каталоги. Таким образом, учащиеся на практике наблюдают как сокращение объёма чтения за счёт выбора столбцов, так и сокращение числа обрабатываемых файлов за счёт партиционирования |
| **Что упрощается относительно университетской лабораторной** | Не используется полноценный кластер Spark — работа выполняется в Google Colab. Данные записываются в локальную файловую систему Colab. Объём данных значительно меньше промышленного, но достаточен для демонстрации различий между CSV и Parquet, работы со столбцами и партициями|
| **Алгоритм действий обучающегося (4–6 шагов)** | обучающегося (4–6 шагов)	1. Получить учебный набор телеметрии и сохранить его в CSV, Parquet и Parquet с партиционированием по robot_id (упр. 1.1–1.3). 2. Измерить размер CSV и Parquet и рассчитать коэффициент уменьшения объёма (упр. 2.1). 3. Сравнить время чтения CSV и Parquet по нескольким повторам и проверить одинаковое количество полученных записей (упр. 3.1–3.3).4. Сравнить чтение только temperature и чтение temperature вместе с payload, чтобы увидеть влияние выбора столбцов на время обработки (упр. 4.2). 5. Выполнить запрос по одному robot_id к партиционированным данным и проверить по плану выполнения, что ненужные партиции не читаются (упр. 4.1). 6. Сравнить варианты партиционирования и сформулировать вывод о преимуществах и ограничениях Parquet и партиционирования (упр. 4.3–4.4) |
| **Измеримый результат / метрика** | Ученик получает конкретные числовые результаты: размер CSV и Parquet в MiB и коэффициент уменьшения объёма (упр. 2.1); время чтения данных и ускорение Parquet относительно CSV (упр. 3.1–3.2); время чтения temperature отдельно и вместе с payload (упр. 4.2); количество файлов и время чтения при разных вариантах партиционирования (упр. 4.3). |
| **Критерий успешного выполнения** |ученик должен корректно сохранить и сравнить данные в CSV и Parquet, получить одинаковое количество записей при одинаковых запросах (упр. 3.2), определить размеры файлов и коэффициент уменьшения объёма (упр. 2.1), сравнить время чтения, показать чтение только нужных столбцов (упр. 4.2) и выбор нужной партиции по robot_id (упр. 4.1). В итоговом выводе ученик объясняет полученные результаты и указывает ограничения Parquet и партиционирования, например проблему большого числа мелких файлов (упр. 4.3–4.4) |
| **Вариант усложнения для НТО / хакатона** | Вариант усложнения для НТО / хакатона: увеличить объём телеметрии и предложить участникам самостоятельно выбрать наиболее эффективный способ хранения данных. Они могут сравнить разные способы сжатия — Snappy, GZIP и ZSTD — по размеру файлов и времени записи (упр. 2.3), проверить разные варианты партиционирования и влияние большого числа мелких файлов (упр. 4.3), а затем выбрать вариант с лучшим соотношением размера и скорости чтения и обосновать свой выбор |
| **Риски и ограничения** |1. Время выполнения зависит от среды Google Colab и файлового кэша, поэтому важна не абсолютная величина, а корректность методики измерения. 2. Колонка payload с SHA-256 почти не сжимается, поэтому специально показывает, что преимущество Parquet зависит от структуры данных. 3. В Colab используется локальный Spark без полноценного кластера, поэтому полученные результаты нельзя напрямую переносить на распределённые промышленные системы  |

---
## Итоговая самопроверка

In [ ]:
checks = {
    "1.1 CSV записан":          CSV_DIR.exists(),
    "1.2 Parquet записан":      len(list(PARQUET_DIR.glob("*.parquet"))) > 0,
    "1.3 partitionBy":          len(part_dirs) == 12,
    "1.4 схема партиции":       "TODO" not in ANSWER_1_4,
    "2.1 размеры и C":          compression_ratio > 1.0,
    "2.2 энтропия":             ratio_nopay > compression_ratio and "TODO" not in ANSWER_2_2,
    "2.3 кодеки":               set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"},
    "2.4 выбор кодека":         "TODO" not in ANSWER_2_4,
    "3.2 бенчмарк чтения":      csv_n == pq_n == part_n,
    "3.3 честное сравнение":    csv_schema_t < csv_t and "TODO" not in ANSWER_3_3,
    "4.1 partition pruning":    "TODO" not in ANSWER_4_1,
    "4.2 column pruning":       t_all > t_one and "TODO" not in ANSWER_4_2,
    "4.3 мелкие файлы":         n_many > n_few,
    "4.4 технический вывод":    "TODO" not in ANSWER_4_4,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

OK   1.1 CSV записан
OK   1.2 Parquet записан
OK   1.3 partitionBy
OK   1.4 схема партиции
OK   2.1 размеры и C
OK   2.2 энтропия
OK   2.3 кодеки
OK   2.4 выбор кодека
OK   3.2 бенчмарк чтения
OK   3.3 честное сравнение
OK   4.1 partition pruning
OK   4.2 column pruning
OK   4.3 мелкие файлы
OK   4.4 технический вывод

Выполнено: 14 из 14
TEACH CARD проверяется преподавателем вручную.


In [ ]:
df.unpersist()
spark.stop()
print("SparkSession остановлена.")